# Leçon 09 - Patron de Conception Métacognition


## Configuration

Ce carnet démontre le modèle de conception Métacognition en utilisant le Microsoft Agent Framework.

**Prérequis :**
- Déploiement Azure OpenAI configuré via des variables d'environnement
- Azure CLI authentifié (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Qu'est-ce que la métacognition ?

La métacognition est **penser à la pensée**. Dans le contexte des agents IA, cela signifie construire des agents qui peuvent :

- **S'auto-réfléchir** sur leurs propres résultats et processus de raisonnement
- **Détecter les erreurs** et se rétablir gracieusement au lieu d’échouer silencieusement
- **Évaluer** si leurs réponses sont complètes et utiles
- **Adapter** leur stratégie lorsqu'une approche initiale ne fonctionne pas (par exemple, revenir à un système de secours)

Un agent métacognitif ne se contente pas de répondre aux questions — il surveille ses propres performances et s’ajuste en temps réel.


## Outils principaux et de secours

Un schéma de métacognition courant est la **stratégie de repli**. L'agent essaie d'abord un outil principal ; s'il échoue (par exemple, une erreur 404), l'agent reconnaît l'échec et bascule de manière transparente vers un outil de secours.

Cela reflète les systèmes du monde réel où les services principaux peuvent être indisponibles et les agents doivent auto-diagnostiquer le problème avant de choisir une voie alternative.

Ci-dessous, nous définissons deux outils de recherche de vols :
- **Principal** — couvre Paris, Tokyo et Barcelone
- **Secours** — couvre Berlin, Sydney et New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agent d'auto-réflexion avec récupération d'erreur

L'agent ci-dessous est chargé d'essayer d'abord le système de vol principal, de reconnaître les échecs, et de basculer de manière transparente vers le système de secours. Après chaque réponse, il s'auto-évalue brièvement pour déterminer s'il a pleinement répondu à la question de l'utilisateur.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Modèle d'Auto-évaluation

Une autre facette de la métacognition est **l'auto-évaluation** : un agent distinct (ou le même agent lors d'une seconde passe) examine une réponse pour sa complétude, son exactitude et son utilité.

Ci-dessous, nous créons un agent `ResponseEvaluator` qui évalue les réponses des agents de voyage selon trois dimensions.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Résumé

Dans cette leçon, vous avez appris à construire des **agents métacognitifs** en utilisant le Microsoft Agent Framework :

- **Autoréflexion** : Agents qui surveillent leur propre raisonnement et communiquent de manière transparente ce qui s'est passé.
- **Récupération d'erreur avec solutions de repli** : Un modèle outil principal + secours où l'agent détecte les échecs (par exemple, erreurs 404) et essaie automatiquement une source alternative.
- **Auto-évaluation** : Un agent évaluateur séparé qui note les réponses en fonction de leur exhaustivité, exactitude et utilité.

Ces modèles rendent les agents plus robustes, transparents et fiables — des qualités cruciales pour les déploiements en production.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d'origine doit être considéré comme la source faisant foi. Pour les informations critiques, une traduction professionnelle humaine est recommandée. Nous ne pouvons être tenus responsables de tout malentendu ou mauvaise interprétation résultant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
